In [ ]:
!pip install selenium
!mkdir -p data
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  apparmor chromium-browser libfuse3-3 liblzo2-2 libudev1 snapd squashfs-tools systemd-hwe-hwdb
  udev
Suggested packages:
  apparmor-profiles-extra apparmor-utils fuse3 zenity | kdialog
The following NEW packages will be installed:
  apparmor chromium-browser chromium-chromedriver libfuse3-3 liblzo2-2 snapd squashfs-tools
  systemd-hwe-hwdb udev
The following packages will be upgraded:
  libudev1
1 upgraded, 9 newly installed, 0 to remove and 48 not upgraded.
Need to get 28.5 MB of archives.
After this operation, 118 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-upd

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import json

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=chrome_options)

MAGAZINE_NAME = "vietnamnet"
HOME_PAGE = "https://vietnamnet.vn/"

driver.get(HOME_PAGE)

menu = driver.find_element(by=By.CLASS_NAME, value="btn-hamburger")
menu.click()

cats = []

cat_list = driver.find_element(by=By.CLASS_NAME, value="header_submenu-content-list")

cat_menus = cat_list.find_elements(by=By.TAG_NAME, value="li")

for cat_menu in cat_menus:
  try:
    a_tag = cat_menu.find_element(by=By.TAG_NAME, value="a")
    cat = a_tag.get_attribute("title").strip()
    href = a_tag.get_attribute("href").strip()
    cats.append({"cat_name": cat, "url": href})
  except Exception as e:
    continue

cats, len(cats)

([{'cat_name': 'premium vietnamnet', 'url': 'https://vietnamnet.vn/premium'},
  {'cat_name': 'Chính trị', 'url': 'https://vietnamnet.vn/thoi-su/chinh-tri'},
  {'cat_name': 'Thời sự', 'url': 'https://vietnamnet.vn/thoi-su'},
  {'cat_name': 'Kinh doanh', 'url': 'https://vietnamnet.vn/kinh-doanh'},
  {'cat_name': 'Thể thao', 'url': 'https://vietnamnet.vn/the-thao'},
  {'cat_name': 'Thế giới', 'url': 'https://vietnamnet.vn/the-gioi'},
  {'cat_name': 'Giáo dục', 'url': 'https://vietnamnet.vn/giao-duc'},
  {'cat_name': 'Giải trí', 'url': 'https://vietnamnet.vn/giai-tri'},
  {'cat_name': 'Văn hóa', 'url': 'https://vietnamnet.vn/van-hoa'},
  {'cat_name': 'Đời sống', 'url': 'https://vietnamnet.vn/doi-song'},
  {'cat_name': 'Sức khỏe', 'url': 'https://vietnamnet.vn/suc-khoe'},
  {'cat_name': 'Thông tin và Truyền thông',
   'url': 'https://vietnamnet.vn/thong-tin-truyen-thong'},
  {'cat_name': 'Pháp luật', 'url': 'https://vietnamnet.vn/phap-luat'},
  {'cat_name': 'Ô tô xe máy', 'url': 'https://vi

In [ ]:
driver.close()

In [ ]:
NUM_ARTICLES_PER_CAT = 500

DATA_URL_FILE = "data/vietnamnet_url.json"

EXCLUDING_CATEGORIES = ["premium vietnamnet", "Hành trình Việt Nam", "English", "Đính chính"]
chrome_options.page_load_strategy = "normal"
driver = webdriver.Chrome(options=chrome_options)

In [ ]:
driver.get(cats[3]['url'])
article_title = driver.find_elements(by=By.CLASS_NAME, value="vnn-title")
url_new = article_title[0].find_element(by=By.TAG_NAME, value="a").get_attribute("href")
url_new
url_new.startswith(HOME_PAGE)
page=driver.find_element(by=By.CLASS_NAME, value="pagination__list")
page.find_elements(by=By.TAG_NAME, value="li")[-1].find_element(by=By.TAG_NAME, value="a").get_attribute("href")
print(page.find_elements(by=By.TAG_NAME, value="li")[-1].find_element(by=By.TAG_NAME, value="a").get_attribute("href"))
print(url_new)
driver.close()

https://vietnamnet.vn/kinh-doanh-page1
https://vietnamnet.vn/lai-suat-ngan-hang-hom-nay-27-8-2024-lai-cao-chi-danh-cho-tien-gui-sieu-khung-2315845.html


In [ ]:
crawled_urls = set()
crawling_log = []

def crawl_each_category_url(driver, category_url):
    """
    Functions cho lấy urls cho từng category sau khi thử nghiệm
    """
    all_urls = set()
    url = category_url
    while len(all_urls) < NUM_ARTICLES_PER_CAT:
        try:
            driver.get(url)
            crawling_log.append(f"Crawling page: {url}")
        except TimeoutException:
            print(f"Timeout loading page: {url}. Skipping this page...")
            break  # Skip to the next page in the pagination

        try:
            WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "vnn-title")))
            article_titles = driver.find_elements(by=By.CLASS_NAME, value="vnn-title")
            for title in article_titles:
                try:
                    url_new = title.find_element(by=By.TAG_NAME, value="a").get_attribute("href")
                    if url_new.startswith(HOME_PAGE) and url_new not in crawled_urls:
                        all_urls.add(url_new)
                        crawled_urls.add(url_new)
                        crawling_log.append(f"Added URL: {url_new}")
                except (StaleElementReferenceException, NoSuchElementException):
                    print(f"Error extracting URL from an article on page: {url}. Skipping this article...")
                    continue  # Skip to the next article on the same page

        except (TimeoutException, StaleElementReferenceException, NoSuchElementException):
            print(f"Error processing page: {url}. Skipping to the next page...")
            break  # Skip to the next page in the pagination

        try:
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "pagination__list")))
            page = driver.find_element(by=By.CLASS_NAME, value="pagination__list")
            url = page.find_elements(by=By.TAG_NAME, value="li")[-1].find_element(by=By.TAG_NAME, value="a").get_attribute("href")
        except (TimeoutException, StaleElementReferenceException, NoSuchElementException):
            print(f"Error finding pagination on page: {url}. Continuing to the next category...")
            break  # Move to the next category if pagination is not found

    return all_urls


saved_cats = {}

for cat in cats:
    cat_name = cat["cat_name"]
    url = cat["url"]
    if cat_name not in EXCLUDING_CATEGORIES:
        print(f"You are at {cat}.")
        urls = crawl_each_category_url(driver, url)
        saved_cats[cat_name] = list(urls)

with open(DATA_URL_FILE, "w") as fOut:
    json.dump(saved_cats, fOut, ensure_ascii=False, indent=4)

driver.close()

print("\nCrawling Log:")
for entry in crawling_log:
    print(entry)

Streaming output truncated to the last 5000 lines.
Added URL: https://vietnamnet.vn/chay-phong-khach-san-khach-chat-vat-tim-cho-xem-chung-ket-phao-hoa-o-da-nang-2300169.html
Crawling page: https://vietnamnet.vn/du-lich-page11
Added URL: https://vietnamnet.vn/khach-san-nishiyama-onsen-keiunkan-hoat-dong-khong-ngung-nghi-suot-hon-1-300-nam-2299826.html
Added URL: https://vietnamnet.vn/uong-coc-ca-phe-gia-nua-trieu-o-tphcm-khach-han-hai-huoc-khen-nhu-o-colombia-2298483.html
Added URL: https://vietnamnet.vn/ket-qua-kiem-tra-quan-vit-co-gioi-de-canh-bon-cau-tai-quang-ninh-2300119.html
Added URL: https://vietnamnet.vn/da-nang-bat-tay-cung-6-tinh-viet-bac-tang-trai-nghiem-cho-du-khach-2299565.html
Added URL: https://vietnamnet.vn/canh-bao-mao-danh-khach-san-o-cua-lo-lua-lay-tien-dat-coc-phong-cua-du-khach-2299883.html
Added URL: https://vietnamnet.vn/tron-nang-nong-voi-dia-diem-cam-trai-xanh-mat-cuc-chua-lanh-ngay-gan-ha-noi-2269186.html
Added URL: https://vietnamnet.vn/du-khach-co-the-bi-pha

In [ ]:
len(crawled_urls)

11719

In [ ]:
FILE_URL_PATH = "data/vietnamnet_url.json"
MAX_ARTICLES_PER_CAT = 500
DATA_FOLDER_OUTPUT = "data/vietnamnet"

!mkdir -p $DATA_FOLDER_OUTPUT

chrome_options.page_load_strategy = "eager"

with open(FILE_URL_PATH, "r") as fIn:
    url_data = json.load(fIn)

len(url_data)

26

In [ ]:
driver = webdriver.Chrome(options=chrome_options)

SAMPLE_ARTICLE_URLS = [
    "https://vietnamnet.vn/vinh-phuc-tich-cuc-chuyen-doi-so-trong-ket-noi-cung-cau-lao-dong-2310383.html",
    "https://vietnamnet.vn/hop-tac-lien-bien-gioi-tang-tinh-doan-ket-huu-nghi-giua-hai-nuoc-viet-lao-2307957.html",
    "https://vietnamnet.vn/bill-gates-canh-bao-viec-tre-em-dung-smartphone-qua-muc-2315659.html",
    "https://vietnamnet.vn/phat-dong-chuong-trinh-binh-chon-ton-vinh-guong-sang-phap-luat-lan-3-2297227.html",
    "https://vietnamnet.vn/nam-dinh-phat-trien-co-so-ha-tang-san-sang-cung-ung-phuc-vu-san-xuat-kinh-doanh-2303421.html"
]

SAMPLE_ARTICLE_URL = SAMPLE_ARTICLE_URLS[2]

In [ ]:
driver.get(SAMPLE_ARTICLE_URL)

In [ ]:
is_emagazine = False
try:
    driver.find_element(by=By.CLASS_NAME, value="emagazine")
except NoSuchElementException:
    is_emagazine = True

is_emagazine

True

In [ ]:
driver.find_element(by=By.CSS_SELECTOR, value="h1.content-detail-title").text

"Phát động chương trình bình chọn, tôn vinh 'Gương sáng pháp luật' lần 3"

In [ ]:
description = driver.find_element(by=By.CLASS_NAME, value="content-detail-sapo").text

VIETNAMNET_PHRASE = "(Vietnamnet) - "
if VIETNAMNET_PHRASE in description:
    description = description.replace(VIETNAMNET_PHRASE, "")

description

'Đồng sáng lập Microsoft Bill Gates cảnh báo việc sử dụng smartphone một cách vô trách nhiệm, nhấn mạnh tác động tiêu cực lên trẻ em.'

In [ ]:
lis_cat = driver.find_element(by=By.CLASS_NAME, value="sm-show-time").find_element(by=By.TAG_NAME, value="ul").find_elements(by=By.TAG_NAME, value="li")
main_cat = lis_cat[1].text if len(lis_cat) > 1 else None
sub_cat = lis_cat[2].text if len(lis_cat) > 2 else None
main_cat, sub_cat

('THÔNG TIN VÀ TRUYỀN THÔNG', 'CÔNG NGHỆ')

In [ ]:
publish_date = driver.find_element(by=By.CLASS_NAME, value='bread-crumb-detail__time').text.strip()
publish_date

'thứ hai, 26/08/2024 - 15:20'

In [ ]:
contents = []
article = driver.find_element(by=By.CLASS_NAME, value="main-content")
children = article.find_elements(by=By.XPATH, value="./*")
children

[<selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f203bca6", element="f.DB6640DF1219F0D66919A8E529394CE6.d.1F089D02B4BF163B18B62DE9451D1408.e.66")>,
 <selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f203bca6", element="f.DB6640DF1219F0D66919A8E529394CE6.d.1F089D02B4BF163B18B62DE9451D1408.e.67")>,
 <selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f203bca6", element="f.DB6640DF1219F0D66919A8E529394CE6.d.1F089D02B4BF163B18B62DE9451D1408.e.68")>,
 <selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f203bca6", element="f.DB6640DF1219F0D66919A8E529394CE6.d.1F089D02B4BF163B18B62DE9451D1408.e.69")>,
 <selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f203bca6", element="f.DB6640DF1219F0D66919A8E529394CE6.d.1F089D02B4BF163B18B62DE9451D1408.e.70")>,
 <selenium.webdriver.remote.webelement.WebElement (session="28b67293ea5f99f392c5a8f2f

In [ ]:
author = "Unknown"
try:
  author = driver.find_element(by=By.CLASS_NAME, value='article-detail-author__info')
  name = author.find_element(by=By.TAG_NAME, value='a').text
  author = name.strip()
  if author == '':
    author = "Unknown"
except:
    pass

author

'Du Lam'

In [ ]:
contents = []
article = driver.find_element(by=By.CLASS_NAME, value="main-content")
children = article.find_elements(by=By.XPATH, value="./*")
content_elements = driver.find_element(by=By.CLASS_NAME, value="main-content").find_elements(by=By.TAG_NAME, value='p')
contents = [element.text.strip() for element in content_elements]

contents

['Smartphone đã trở thành một phần không thể thiếu trong cuộc sống của nhiều người, từ văn phòng về nhà, từ trường học đến sân chơi. Bên cạnh những lợi ích to lớn, smartphone cũng có thể mang đến rắc rối nếu không biết cách kiểm soát.',
 'Bill Gates liên tục cảnh báo việc sử dụng công nghệ quá mức, nhấn mạnh trẻ em nên dùng smartphone vừa phải và đúng cách.',
 'Trong cuộc phỏng vấn với tạp chí The Mirror, đồng sáng lập Microsoft cho rằng, công nghệ nên phục vụ con người, không phải kiểm soát họ. Khi dùng điện thoại quá nhiều, có thể dẫn đến các vấn đề sức khỏe tinh thần, cô lập xã hội và khó tập trung.',
 'Theo Gates, smartphone không nên chi phối cuộc sống của chúng ta, đặc biệt trong những khoảnh khắc quan trọng với người thân. Ông khuyên tránh sử dụng điện thoại trong bữa ăn để tập trung nói chuyện, trao đổi và hiện diện cùng với gia đình.',
 'Bản thân ông cũng áp dụng các nguyên tắc này tại nhà của mình, hạn chế con dùng thiết bị cho đến năm 14 tuổi, dù con ông phàn nàn rằng bạn bè

In [ ]:
driver.close()

InvalidSessionIdException: Message: invalid session id
Stacktrace:
#0 0x59f5f4d7581a <unknown>
#1 0x59f5f4a43c91 <unknown>
#2 0x59f5f4a852a3 <unknown>
#3 0x59f5f4ab8374 <unknown>
#4 0x59f5f4ab27fd <unknown>
#5 0x59f5f4ab19a5 <unknown>
#6 0x59f5f4a0e0a8 <unknown>
#7 0x59f5f4d3ca7b <unknown>
#8 0x59f5f4d40a31 <unknown>
#9 0x59f5f4d28645 <unknown>
#10 0x59f5f4d415a2 <unknown>
#11 0x59f5f4d0d81f <unknown>
#12 0x59f5f4a0cad1 <unknown>
#13 0x7d7c99036d90 <unknown>


In [ ]:
class MagazineArticleException(Exception):
    pass

def get_content_metadata(driver, article_url):

    """
    Extracts and returns metadata and content from a given article URL.

    :param driver: Selenium WebDriver instance.
    :param article_url: URL of the article to extract data from.
    :return: Dictionary containing article metadata and content.
    """

    # Get to current article
    driver.get(article_url)


    # Tạm thời bỏ qua emagazine
    is_emagazine = True
    try:
        driver.find_element(by=By.CLASS_NAME, value="emagazine")
    except NoSuchElementException:
        is_emagazine = False

    if is_emagazine:
        raise MagazineArticleException("We ignore MagazineArticle")

    title = driver.find_element(by=By.CSS_SELECTOR, value="h1.content-detail-title").text

    description = driver.find_element(by=By.CLASS_NAME, value="content-detail-sapo").text

    VIETNAMNET_PHRASE = "(Vietnamnet) - "
    if VIETNAMNET_PHRASE in description:
        description = description.replace(VIETNAMNET_PHRASE, "")

    lis_cat = driver.find_element(by=By.CLASS_NAME, value="sm-show-time").find_element(by=By.TAG_NAME, value="ul").find_elements(by=By.TAG_NAME, value="li")
    main_cat = lis_cat[1].text if len(lis_cat) > 1 else None
    sub_cat = lis_cat[2].text if len(lis_cat) > 2 else None

    publish_date = driver.find_element(by=By.CLASS_NAME, value='bread-crumb-detail__time').text.strip()

    author = "Unknown"
    try:
      author = driver.find_element(by=By.CLASS_NAME, value='article-detail-author__info')
      name = author.find_element(by=By.TAG_NAME, value='a').text
      author = name.strip()
      if author == '':
          author = "Unknown"
    except:
          pass

    contents = []
    article = driver.find_element(by=By.CLASS_NAME, value="main-content")
    children = article.find_elements(by=By.XPATH, value="./*")
    content_elements = driver.find_element(by=By.CLASS_NAME, value="main-content").find_elements(by=By.TAG_NAME, value='p')
    contents = [element.text.strip() for element in content_elements]

    return {
        "url": article_url,
        "title": title,
        "description": description,
        "content": "\n".join(contents),
        "metadata": {
            "cat": main_cat,
            "subcat": sub_cat,
            "published_date": publish_date,
            "author": author
        }
    }


In [ ]:
driver = webdriver.Chrome(options=chrome_options)

all_cat_data = []  # Initialize a list to store all categories' data

for cat, urls in url_data.items():

    print(f"Thu thập dữ liệu thể loại {cat} ..")
    count_crawled = 0
    cat_data = []
    for url in urls:
        try:
            cat_data.append(get_content_metadata(driver, url))
            count_crawled += 1
            if MAX_ARTICLES_PER_CAT and count_crawled >= MAX_ARTICLES_PER_CAT:
                break
        except (StaleElementReferenceException) as e:
            print(f"Bug at url: {url}, with StaleElementReferenceException")
            driver.refresh()
            continue
        except (NoSuchElementException) as e:
            print(f"Bug at url: {url}, with NoSuchElementException")
            driver.refresh()
            continue
        except MagazineArticleException as e:
            continue

    all_cat_data.extend(cat_data)  # Add this category's data to the main list

driver.close()

# Dump all categories' data into a single file
with open(os.path.join(DATA_FOLDER_OUTPUT, "vietnamnet_processed.json"), "w") as fOut:
    json.dump(all_cat_data, fOut, ensure_ascii=False, indent=4)

Thu thập dữ liệu thể loại Chính trị ..
Bug at url: https://vietnamnet.vn/tieu-su-tan-pho-thu-tuong-chinh-phu-ho-duc-phoc-2315570.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/ong-mai-van-chinh-tan-truong-ban-dan-van-trung-uong-2314203.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/tieu-su-tong-bi-thu-nguyen-phu-trong-2303748.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/tieu-su-uy-vien-bo-chinh-tri-bo-truong-bo-cong-an-luong-tam-quang-2312484.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/ong-do-duc-duy-tu-bi-thu-yen-bai-den-bo-truong-tai-nguyen-va-moi-truong-2315580.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/chan-dung-pho-thu-tuong-chinh-phu-bo-truong-ngoai-giao-bui-thanh-son-2315561.html, with NoSuchElementException
Bug at url: https://vietnamnet.vn/danh-sach-26-thanh-vien-chinh-phu-nhiem-ky-2021-2026-2315886.html, with NoSuchElementException
Bug at url: https://vietnamnet.v

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
FOLDER_SAVED_GOOGLE_COLAB = "/content/drive/MyDrive/vietnamnet_processed/"

# Copy
!cp -r data $FOLDER_SAVED_GOOGLE_COLAB